In [ ]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from PIL import Image
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, confusion_matrix
from tqdm import tqdm

In [ ]:
# Settings
# ==========================================
IMAGE_DIR = r"D:\COURSE_DATA\Intro_Deep_Learning\project\data\Post_Impressionism"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

#Best known hyper parameters

LEARNING_RATE = 6.01e-5
BATCH_SIZE = 32
OPTIMIZER_NAME = 'Adam'
NUM_EPOCHS = 5

print(f"✅ Device: {DEVICE}")
print(f"🚀 Starting Final Training for ALEXNET with: LR={LEARNING_RATE}, Batch={BATCH_SIZE}, Epochs={NUM_EPOCHS}")

✅ Device: cuda
🚀 Starting Final Training for ALEXNET with: LR=6.01e-05, Batch=32, Epochs=5


In [ ]:
#Data preparation
# ==========================================
def prepare_data(dir_path):
    if not os.path.exists(dir_path):
        print(f"❌ Error: Folder not found at {dir_path}")
        exit()

    all_files = [f for f in os.listdir(dir_path) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
    labels = [1 if f.lower().startswith("vincent-van-gogh") else 0 for f in all_files]

    print(f"📂 Total images found: {len(all_files)}")

    #  Train (80%), Test (10%), Validation (10%)
    X_train_val, X_test, y_train_val, y_test = train_test_split(
        all_files, labels, test_size=0.10, random_state=42, stratify=labels
    )

    X_train, X_val, y_train, y_val = train_test_split(
        X_train_val, y_train_val, test_size=0.1111, random_state=42, stratify=y_train_val
    )

    return X_train, y_train, X_val, y_val, X_test, y_test


X_train, y_train, X_val, y_val, X_test, y_test = prepare_data(IMAGE_DIR)
print(f"📊 Split: Train={len(X_train)} | Val={len(X_val)} | Test={len(X_test)}")

📂 Total images found: 6450
📊 Split: Train=5160 | Val=645 | Test=645


In [ ]:
#3. Dataset & Transforms
#==========================================
class SimpleFolderDataset(Dataset):
    def __init__(self, filenames, labels, root_dir, transform=None):
        self.filenames = filenames
        self.labels = labels
        self.root_dir = root_dir
        self.transform = transform

    def __len__(self):
        return len(self.filenames)

    def __getitem__(self, idx):
        img_path = os.path.join(self.root_dir, self.filenames[idx])
        try:
            image = Image.open(img_path).convert('RGB')
        except:
            image = Image.new('RGB', (224, 224), (0, 0, 0))

        if self.transform:
            image = self.transform(image)
        return image, self.labels[idx]

In [ ]:

# Image transformation
train_transforms = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

val_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

In [ ]:
# Data loading
train_loader = DataLoader(SimpleFolderDataset(X_train, y_train, IMAGE_DIR, train_transforms),
                          batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(SimpleFolderDataset(X_val, y_val, IMAGE_DIR, val_transforms),
                        batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(SimpleFolderDataset(X_test, y_test, IMAGE_DIR, val_transforms),
                         batch_size=BATCH_SIZE, shuffle=False)

In [ ]:
# Constructing AlexNet model
# ==========================================
def get_model():
    print("🏗️  Building AlexNet...")
    # AlexNet loading
    model = models.alexnet(weights=models.AlexNet_Weights.IMAGENET1K_V1)

    #Features freezing
    for param in model.features.parameters():
        param.requires_grad = False

    # Setting classifier layer
    num_ftrs = model.classifier[6].in_features
    model.classifier[6] = nn.Linear(num_ftrs, 2)

    return model.to(DEVICE)


model = get_model()

🏗️  Building AlexNet...


In [ ]:
#Setting loss weights
class_weights = torch.tensor([1.0, 5.0]).to(DEVICE)
criterion = nn.CrossEntropyLoss(weight=class_weights)

# Optimizer setting
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

In [ ]:
#Training process
# ==========================================
best_f1 = 0.0

for epoch in range(NUM_EPOCHS):
    print(f"\n--- Epoch {epoch + 1}/{NUM_EPOCHS} ---")

    # Training
    model.train()
    train_loss = 0.0
    loop = tqdm(train_loader, desc="Training")

    for imgs, lbls in loop:
        imgs, lbls = imgs.to(DEVICE), lbls.to(DEVICE)

        optimizer.zero_grad()
        outputs = model(imgs)
        loss = criterion(outputs, lbls)
        loss.backward()
        optimizer.step()

        train_loss += loss.item()
        loop.set_postfix(loss=loss.item())

    # Validation
    model.eval()
    all_preds, all_labels, all_probs = [], [], []

    with torch.no_grad():
        for imgs, lbls in tqdm(val_loader, desc="Validating"):
            imgs, lbls = imgs.to(DEVICE), lbls.to(DEVICE)
            outputs = model(imgs)
            probs = torch.softmax(outputs, dim=1)[:, 1]
            _, preds = torch.max(outputs, 1)

            all_probs.extend(probs.cpu().numpy())
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(lbls.cpu().numpy())

    #Scores calculation
    val_f1 = f1_score(all_labels, all_preds, average='binary')
    val_acc = accuracy_score(all_labels, all_preds)
    try:
        val_auc = roc_auc_score(all_labels, all_probs)
    except:
        val_auc = 0.5
    cm = confusion_matrix(all_labels, all_preds)

    print(f"📊 Results: F1: {val_f1:.4f} | Acc: {val_acc:.4f} | AUC: {val_auc:.4f}")
    print(f"🔲 Confusion Matrix: {cm.tolist()}")

    #Saving the model
    if val_f1 > best_f1:
        best_f1 = val_f1
        torch.save(model.state_dict(), "best_vangogh_alexnet_final.pth")
        print("💾 Model Saved (Best F1)")

print("\n✅ Training Complete!")


--- Epoch 1/5 ---


Validating: 100%|██████████| 21/21 [01:07<00:00,  3.20s/it]


📊 Results: F1: 0.7317 | Acc: 0.8977 | AUC: 0.9581
🔲 Confusion Matrix: [[489, 55], [11, 90]]
💾 Model Saved (Best F1)

--- Epoch 2/5 ---


Validating: 100%|██████████| 21/21 [00:45<00:00,  2.15s/it]


📊 Results: F1: 0.7333 | Acc: 0.9008 | AUC: 0.9631
🔲 Confusion Matrix: [[493, 51], [13, 88]]
💾 Model Saved (Best F1)

--- Epoch 3/5 ---


...

📊 Results: F1: 0.8447 | Acc: 0.9504 | AUC: 0.9725
🔲 Confusion Matrix: [[526, 18], [14, 87]]
💾 Model Saved (Best F1)

--- Epoch 4/5 ---


Validating: 100%|██████████| 21/21 [00:38<00:00,  1.86s/it]


📊 Results: F1: 0.7946 | Acc: 0.9287 | AUC: 0.9714
🔲 Confusion Matrix: [[510, 34], [12, 89]]

--- Epoch 5/5 ---


Validating: 100%|██████████| 21/21 [00:32<00:00,  1.54s/it]

📊 Results: F1: 0.8000 | Acc: 0.9333 | AUC: 0.9679
🔲 Confusion Matrix: [[516, 28], [15, 86]]

✅ Training Complete!


In [ ]:
# Testing and scores
# ==========================================
print("\n🔍 Running Final Evaluation on Test Set...")

# טעינת המודל מחדש מהקובץ שנשמר
model.load_state_dict(torch.load("best_vangogh_alexnet_final.pth"))
model.eval()

test_preds, test_labels, test_probs = [], [], []

with torch.no_grad():
    for imgs, lbls in tqdm(test_loader, desc="Testing"):
        imgs, lbls = imgs.to(DEVICE), lbls.to(DEVICE)
        outputs = model(imgs)
        probs = torch.softmax(outputs, dim=1)[:, 1]
        _, preds = torch.max(outputs, 1)

        test_probs.extend(probs.cpu().numpy())
        test_preds.extend(preds.cpu().numpy())
        test_labels.extend(lbls.cpu().numpy())

test_f1 = f1_score(test_labels, test_preds, average='binary')
test_acc = accuracy_score(test_labels, test_preds)
test_auc = roc_auc_score(test_labels, test_probs)
test_cm = confusion_matrix(test_labels, test_preds)

print("\n🏆 FINAL TEST RESULTS (ALEXNET):")
print(f"   F1 Score: {test_f1:.4f}")
print(f"   Accuracy: {test_acc:.4f}")
print(f"   AUC-ROC:  {test_auc:.4f}")
print(f"   Confusion Matrix:\n{test_cm}")


🔍 Running Final Evaluation on Test Set...


Testing: 100%|██████████| 21/21 [00:36<00:00,  1.72s/it]


🏆 FINAL TEST RESULTS (ALEXNET):
   F1 Score: 0.8148
   Accuracy: 0.9380
   AUC-ROC:  0.9671
   Confusion Matrix:
[[517  28]
 [ 12  88]]
